#Мамоу Асман ИУ5-21М Лаборатораня работа №1

## Импорт библиотек и загрузка датасета

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Загрузка данных (предполагается, что файл уже в среде Colab)
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
print("Информация о датасете:")
print(df.info())
print("\nКоличество пропусков по столбцам:")
print(df.isnull().sum())

Информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB
None

Количество пропусков по столбцам:
id                     0
gender                 0
age                    0
hypertension       

## Предобработка датасета: выбор признаков и разделение

In [ ]:
# 1. Определим целевую переменную
y = df['stroke']

# 2. Удалим ненужные колонки:
#    - 'id' – идентификатор, не несёт полезной информации.
drop_columns = ['id', 'stroke']
X = df.drop(columns=drop_columns)

# 3. Разделим признаки на числовые и категориальные
# Числовые: возраст, уровень глюкозы, ИМТ, гипертония, болезни сердца (бинарные)
num_features = ['age', 'avg_glucose_level', 'bmi', 'hypertension', 'heart_disease']
# Категориальные: пол, семейное положение, тип работы, тип проживания, статус курения
cat_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

# Проверим оставшиеся пропуски (должны быть только в 'bmi')
print("Пропуски в выбранных признаках:")
print(X[num_features + cat_features].isnull().sum())

Пропуски в выбранных признаках:
age                    0
avg_glucose_level      0
bmi                  201
hypertension           0
heart_disease          0
gender                 0
ever_married           0
work_type              0
Residence_type         0
smoking_status         0
dtype: int64


## Пайплайн предобработки: заполнение пропусков, кодирование и нормализация

In [ ]:
# Пайплайн для числовых признаков:
# - заполняем пропуски медианой
# - масштабируем (стандартизация)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Пайплайн для категориальных признаков:
# - заполняем пропуски самым частым значением
# - применяем One-Hot Encoding
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Объединяем преобразователи в ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ],
    remainder='drop'
)

# Применяем preprocessing
X_processed = preprocessor.fit_transform(X)
print(f"Размер обработанных данных: {X_processed.shape}")

Размер обработанных данных: (5110, 21)


## Восстановление имён признаков и создание итогового DataFrame


In [ ]:
# Получаем имена признаков после OneHotEncoder
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(cat_features).tolist()
all_feature_names = num_features + cat_feature_names

# Создаём DataFrame для наглядности
df_processed = pd.DataFrame(X_processed, columns=all_feature_names)
print("Первые 5 строк обработанного датафрейма:")
print(df_processed.head())

# Проверим, что пропусков больше нет
print("\nКоличество пропусков в итоговой матрице:")
print(df_processed.isnull().sum().sum())

Первые 5 строк обработанного датафрейма:
        age  avg_glucose_level       bmi  hypertension  heart_disease  \
0  1.051434           2.706375  1.005086     -0.328602       4.185032   
1  0.786070           2.121559 -0.098981     -0.328602      -0.238947   
2  1.626390          -0.005028  0.472536     -0.328602       4.185032   
3  0.255342           1.437358  0.719327     -0.328602      -0.238947   
4  1.582163           1.501184 -0.631531      3.043196      -0.238947   

   gender_Female  gender_Male  gender_Other  ever_married_No  \
0            0.0          1.0           0.0              0.0   
1            1.0          0.0           0.0              0.0   
2            0.0          1.0           0.0              0.0   
3            1.0          0.0           0.0              0.0   
4            1.0          0.0           0.0              0.0   

   ever_married_Yes  ...  work_type_Never_worked  work_type_Private  \
0               1.0  ...                     0.0                